# Phase 7 - Prior Case Retrieval (PCR) - Architecture 4: Encoder-decoder Transformer

**Task:** for every query case, rank the cases of the matching candidate pool and return the **top 7**.
**Metric:** Micro-F1@7 (also Micro-Precision@7 / Micro-Recall@7).

**Starting point:** the Phase-1 legal language model of Architecture 4 (`best.pt`). Encoder-decoder model. For retrieval we only use the bidirectional ENCODER (the decoder is not needed).
The tokenizer and the architecture are used **exactly as in Phase 1** (the code cells below are copied from the Phase-1 notebook).

## How the retrieval model works (simple version)
1. A long case is cut into **chunks** of `CHUNK_LEN` tokens (at most `MAX_CHUNKS` chunks).
2. The Phase-1 model reads each chunk and we **pool** (average) its token vectors into **one chunk vector**.
3. The chunk vectors of a case are averaged into **one case vector**  (hierarchical document representation).
4. A query and a candidate are compared with **cosine similarity**; the 7 most similar candidates are returned.
5. Training uses a **contrastive loss**: the query vector should be closer to its relevant case than to the other cases
   in the batch (**in-batch negatives**) and than to one **hard negative** (a wrong case that the frozen model finds similar).

## Fine-tuning techniques (all hand-written, taken from `q4_peft.py`; no `peft` library)
Every experiment starts again from the **same Phase-1 weights**, freezes the whole model, and trains only a few new parameters:

| # | Technique | What is trained | Where |
|---|-----------|-----------------|-------|
| 1 | **Soft prompt** (prompt tuning) | `m` learned vectors | in front of the input tokens |
| 2 | **Bottleneck adapter** | tiny down/up MLP | after every block (residual) |
| 3 | **LoRA** | low-rank `A`, `B` | additive update of the attention matrices q, k, v, o |
| 4 | **LoRMA** (LoRMA+) | low-rank `A`, `B` | multiplicative update `W' = (s*B*A + I) W0` of q, k, v, o |

No knowledge distillation is used. A **zero-shot baseline** (frozen Phase-1 model, no training) is reported for reference,
so we can measure how much the PEFT training adds.

## Run order
Run all cells top to bottom. Attach the tokenizer dataset (`kratikasum/tokenizer`) and the dataset that holds the Phase-1 checkpoint
`best.pt` as Kaggle inputs, and add your Hugging Face token as the Kaggle secret `HF`.
Every finished run is saved to its own folder, so if the session stops you can simply re-run the notebook: finished runs are skipped.

## 1. Configuration

### Imports, Hugging Face login, GPU check

In [1]:
import os, random, json, time, math, zipfile, gc, glob, pickle
from typing import List, Dict, Tuple
import numpy as np
import torch
from tqdm.auto import tqdm            # progress bars
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


### Settings (everything you may want to change is in this one cell)

In [2]:
# ---------------- paths ----------------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
DATA_DIR = 'Phase_7_PCR'                         # folder inside the HF dataset: Train/ Dev/ Test/
ARCH_NAME = 'arch4_encdec'
# Phase-1 checkpoint of THIS architecture (the best.pt written by the Phase-1 notebook)
PHASE1_CKPT = glob.glob('/kaggle/input/**/last.pt', recursive=True)[0]
OUTPUT_DIR = f'/kaggle/working/phase7_{ARCH_NAME}'   # one folder per run is created inside it
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- document representation ----------------
CHUNK_LEN = 512        # tokens per chunk (must be <= the Phase-1 max_seq_len, which is 2048)
MAX_CHUNKS = 4         # a case is cut into at most this many chunks -> we read the first CHUNK_LEN*MAX_CHUNKS tokens
MAX_TOKENS = CHUNK_LEN * MAX_CHUNKS
POOLING = 'mean'       # how the token vectors of a chunk become one vector: 'mean' | 'last'
SIMILARITY = 'cosine'  # 'cosine' | 'dot'

# ---------------- training objective ----------------
LOSS = 'infonce'       # 'infonce' (contrastive, softmax over candidates) | 'margin' (pairwise hinge loss)
NEGATIVES = 'hard'     # 'hard' = in-batch negatives + 1 mined hard negative per query | 'inbatch' = in-batch only
TAU = 0.05             # temperature of the contrastive loss (cosine similarity only)
MARGIN = 0.2           # margin of the pairwise loss
HARD_NEG_POOL = 30     # hard negative of a query is drawn from the 30 most similar NON-relevant train candidates

# ---------------- training ----------------
EPOCHS = 2
BATCH_QUERIES = 4      # queries per forward pass. Every query brings 1 positive (+1 hard negative) -> 12 cases = ~48 chunks
GRAD_ACCUM = 4         # effective batch = BATCH_QUERIES * GRAD_ACCUM queries
CHUNK_BATCH_EVAL = 32  # chunks encoded at once while evaluating (lower it if you get CUDA OOM)
MAX_TRAIN_QUERIES = None   # e.g. 200 for a quick test run; None = use all train queries
WARMUP_FRAC = 0.05
WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0
GRAD_CKPT = True       # gradient checkpointing: saves a lot of memory, ~30% slower
SEED = 42
TOP_K = 7              # the competition evaluates the top 7 candidates

# ---------------- PEFT settings ----------------
METHODS = ['soft_prompt', 'adapter', 'lora', 'lorma']      # the 4 techniques of q4_peft
PEFT_LR = {'soft_prompt': 1e-2, 'adapter': 1e-3, 'lora': 5e-4, 'lorma': 5e-4}   # prompts need a much larger LR
SOFT_TOKENS = 16                                  # soft prompt: number of virtual tokens m
SOFT_INIT_TEXT = ' Find prior legal cases with similar facts and legal issues:'   # soft prompt is initialised from these words
ADAPTER_BOTTLENECK = 16                           # adapter width
RANK, ALPHA = 8, 16                               # LoRA / LoRMA rank r and scaling alpha (update is scaled by alpha/r = 2)
TARGETS = ('q', 'k', 'v', 'o')                    # attention matrices adapted by LoRA / LoRMA

# ---------------- extra experiments for the analysis ----------------
RUN_ABLATIONS = True            # pooling / chunking / similarity / loss / negatives ablations (extra LoRA runs)
ABLATION_METHOD = 'lora'        # technique used for the ablations (cheapest of the weight-based methods)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# T4/P100 have no bf16 -> fp16 + GradScaler; Ampere+ -> bf16
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print('device =', device, '| amp dtype =', amp_dtype)

device = cuda | amp dtype = torch.float16


## 2. Tokenizer (unchanged from Phase 1)

### Tokenizer class

In [3]:
class ByteLevelBPE:
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')

### Load the trained tokenizer

In [4]:
cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)
if cands:
    tok_path = cands[0]
else:                                   # dataset not attached as input -> download it
    import kagglehub
    tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')
tokenizer = ByteLevelBPE.load(tok_path)

# The embedding table must cover the largest id, and ids should be unique.
VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
n_unique = len(set(tokenizer.vocab.values()))
print(f'tokenizer entries: {len(tokenizer):,} | unique ids: {n_unique:,} | embedding size: {VOCAB_SIZE:,}')
if n_unique != len(tokenizer):
    print('WARNING: some tokens share an id (can happen when two merges build the same byte string).')

tokenizer entries: 48,000 | unique ids: 48,000 | embedding size: 48,000


### Fast word-level encoder (cached)

In [5]:
word_cache = {}                          # tokenizer works word by word, so caching per word is exact

def fast_encode(text):
    out = []
    for word in text.split():
        if word not in word_cache:
            word_cache[word] = tokenizer.encode(word)
        out.extend(word_cache[word])
    return out

In [6]:
PAD_ID = tokenizer.vocab[b'<pad>']      # used to pad short chunks (and as a placeholder for soft-prompt positions)

## 3. Phase-1 model code (unchanged from Phase 1)

In [7]:
import math
from dataclasses import dataclass
from types import SimpleNamespace
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

### Config

In [8]:
@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_head: int = 12
    n_kv_head: Optional[int] = None
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    n_enc_layer: int = 6
    n_dec_layer: int = 6
    enc_len: int = 768                   # tokens given to the encoder (prompt); the rest is the decoder target
    cross_every: int = 1                 # cross-attention in every k-th decoder layer
    enc_downsample: int = 1              # stride for compressing the encoder memory
    mtp_heads: int = 0
    mtp_weight: float = 0.3

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):
        return int(round(8 / 3 * self.d_model / 64)) * 64

### Building blocks: RMSNorm, RoPE, SwiGLU MLP

In [9]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

### KV cache (new: lets generation process one token per step instead of the whole prefix)

In [10]:
# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)

### Self-attention (GQA / QK-norm / RoPE-or-NoPE / optional value residual, with KV-cache support)

In [11]:
class Attention(nn.Module):
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1)), v_new

### LM head (final norm + tied unembedding + multi-token-prediction heads)

In [12]:
class LMHead(nn.Module):
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)
        self.main_loss = loss.detach()          # next-token loss only (without MTP terms), for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)
        return logits, loss

### Helpers: gradient checkpointing, weight init, base class

In [13]:
def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n

### Cross-attention (decoder -> encoder memory)

In [14]:
class CrossAttention(nn.Module):
    """Decoder->encoder attention (no positional encoding on memory), GQA + optional QK-norm."""

    def __init__(self, cfg: Config):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout

    def forward(self, x, mem, slot=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        if slot is not None and slot.k is not None:              # memory K/V were computed on an earlier step
            k, v = slot.k, slot.v
            if self.q_norm is not None:
                q = self.q_norm(q)
        else:
            S = mem.shape[1]
            k = self.k(mem).view(B, S, self.Hkv, self.hd).transpose(1, 2)
            v = self.v(mem).view(B, S, self.Hkv, self.hd).transpose(1, 2)
            if self.q_norm is not None:
                q, k = self.q_norm(q), self.k_norm(k)
            if slot is not None:
                slot.k, slot.v = k, v
        out = sdpa_gqa(q, k, v, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, -1))

### Encoder block and decoder block

In [15]:
class Block(nn.Module):
    """Bidirectional encoder block."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attn, self.mlp = Attention(cfg, causal=False), MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope):
        h, _ = self.attn(self.n1(x), rope)
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x)))


class DecBlock(nn.Module):
    def __init__(self, cfg: Config, cross: bool):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.self_attn = Attention(cfg, causal=True)
        self.cross = CrossAttention(cfg) if cross else None
        self.nx = RMSNorm(cfg.d_model) if cross else None
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope, mem, cache=None):
        h, _ = self.self_attn(self.n1(x), rope, cache=None if cache is None else cache.self_kv)
        x = x + self.drop(h)
        if self.cross is not None:
            x = x + self.drop(self.cross(self.nx(x), mem, None if cache is None else cache.cross_kv))
        return x + self.drop(self.mlp(self.n2(x)))

### Encoder-decoder model: layers, encoder, decoder

In [16]:
class EncDecBase(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.eval_offset = cfg.enc_len - 1               # logits/labels before this index are the prompt
        self.tok = nn.Embedding(cfg.vocab_size, d)                     # shared by encoder & decoder
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.enc_blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_enc_layer)])
        self.enc_norm = RMSNorm(d)
        ds = cfg.enc_downsample
        self.down = nn.Conv1d(d, d, kernel_size=ds, stride=ds) if ds > 1 else None
        self.dec_blocks = nn.ModuleList(
            [DecBlock(cfg, cross=(i % cfg.cross_every == cfg.cross_every - 1)) for i in range(cfg.n_dec_layer)])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_enc_layer + cfg.n_dec_layer)

    def encode(self, ids):
        x = self.tok(ids)
        rope = self.rope(ids.shape[1])
        for blk in self.enc_blocks:
            x = ckpt_call(blk, self.cfg.grad_ckpt and self.training, x, rope)
        x = self.enc_norm(x)
        if self.down is not None:
            ds = self.cfg.enc_downsample
            x = self.down(x[:, : x.shape[1] // ds * ds].transpose(1, 2)).transpose(1, 2)
        return x

    def decode(self, ids, mem, caches=None, start=0):
        x = self.tok(ids)
        rope = self.rope(ids.shape[1], start)
        for i, blk in enumerate(self.dec_blocks):
            if caches is None:
                x = ckpt_call(blk, self.cfg.grad_ckpt and self.training, x, rope, mem)
            else:
                x = blk(x, rope, mem, cache=caches[i])
        return x

### Encoder-decoder model: training forward + cached generation

In [17]:
class EncDecLM(EncDecBase):
    """training forward + cached generation."""

    def forward(self, idx, targets=None, src=None):
        P = self.cfg.enc_len
        assert idx.shape[1] > P, f"need sequence longer than enc_len={P}"
        mem = self.encode(idx[:, :P])
        h = self.decode(idx[:, P - 1:], mem)               # first decoder input = last prompt token
        labels = targets[:, P - 1:] if targets is not None else None
        return self.head(h, labels)                         # logits/loss over the continuation only

    # ---- generation: encoder reads the prompt once, decoder produces the continuation (cached) ----
    def prefill(self, idx, max_new):
        mem = self.encode(idx[:, -self.cfg.enc_len:])
        caches = [SimpleNamespace(self_kv=KVCache(max_new + 1), cross_kv=SimpleNamespace(k=None, v=None))
                  for _ in self.dec_blocks]
        state = SimpleNamespace(mem=mem, caches=caches, pos=0)
        return self.step(idx[:, -1:], state), state          # decoder's first input = last prompt token

    def step(self, tok, state):
        h = self.decode(tok, state.mem, state.caches, start=state.pos)
        state.pos += tok.shape[1]
        return self.head.logits(h[:, -1])

### Baseline / modified architecture definitions

In [18]:
Model = EncDecLM


BASE = dict(d_model=768, n_head=12, max_seq_len=2048, enc_len=768)


def cfg_baseline(**kw):
    return Config(**{**BASE, "n_enc_layer": 6, "n_dec_layer": 6, "cross_every": 1, **kw})


def cfg_modified(**kw):
    return Config(**{**BASE, "n_enc_layer": 8, "n_dec_layer": 9, "cross_every": 2, "enc_downsample": 2,
                     "n_kv_head": 4, "qk_norm": True, "tie_embeddings": False, "mtp_heads": 2, **kw})

## 4. Load the Phase-1 model

In [19]:
# [CKPT] The checkpoint is read ONCE (on the CPU). `build_phase1_model()` then gives a fresh copy of the
# Phase-1 weights, so every experiment starts from exactly the same model.
PHASE1 = torch.load(PHASE1_CKPT, map_location='cpu', weights_only=False)
cfg = Config(**PHASE1['cfg'])          # the architecture config that was stored by Phase 1
cfg.grad_ckpt = GRAD_CKPT              # only changes HOW the forward pass is computed (memory), not the model itself
assert CHUNK_LEN + SOFT_TOKENS <= cfg.max_seq_len, 'chunk (+ soft prompt) must fit the Phase-1 position table'


def build_phase1_model():
    m = Model(cfg)
    m.load_state_dict(PHASE1['model'])
    return m.to(device)


_m = build_phase1_model()
N_TOTAL_PARAMS = _m.num_params()
del _m
print(f'Loaded Phase-1 {ARCH_NAME}: {N_TOTAL_PARAMS / 1e6:.1f}M parameters (checkpoint step {PHASE1["step"]})')

Loaded Phase-1 arch4_encdec: 189.4M parameters (checkpoint step 1525)


### Architecture-specific part: how do we get one vector per token?

In [20]:
def get_blocks(model):
    """The transformer blocks that LoRA / LoRMA / adapters will touch: the ENCODER blocks only
    (the decoder is not used for retrieval, so adapting it would waste parameters)."""
    return model.enc_blocks


def encode_tokens(model, ids):
    """ids (B, L) -> (hidden (B, L/stride, d), stride).
    The bidirectional encoder reads the whole chunk and gives one vector per token (already RMS-normalised).
    In the 'modified' Phase-1 model the encoder output is down-sampled by `enc_downsample`, so it is shorter."""
    h = model.encode(ids)
    return h, model.cfg.enc_downsample

## 5. Data

Each split has **its own candidate pool** and must only be used with its own queries:
`train.json + train_candidates.json`, `dev.json + dev_candidates.json`, `test.json + test_candidates.json`.
A case's `text` is a list of sentences; we join them and tokenize them with the Phase-1 tokenizer.

In [21]:
# [DATA] download one json file of the Phase-7 dataset
def load_json(name):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f'{DATA_DIR}/{name}', repo_type='dataset', token=HF_TOKEN)
    return json.load(open(path, encoding='utf-8'))


def tokenize_case(rec):
    """case record -> list of token ids. Only the first MAX_TOKENS tokens are kept (long cases are truncated).
    A word needs at least 1 token, so reading only the first MAX_TOKENS words is enough (and much faster)."""
    text = ' '.join(rec['text']) if isinstance(rec['text'], list) else rec['text']
    words = text.split()[:MAX_TOKENS]
    return fast_encode(' '.join(words))[:MAX_TOKENS]


SPLITS = {}
for split, qfile, cfile in [('train', 'Train/train.json', 'Train/train_candidates.json'),
                            ('dev',   'Dev/dev.json',     'Dev/dev_candidates.json'),
                            ('test',  'Test/test.json',   'Test/test_candidates.json')]:
    queries, cands = load_json(qfile), load_json(cfile)
    # we expect {case_id: {"text": [...sentences...], "relevant_candidates": [...]}}
    assert isinstance(queries, dict) and isinstance(cands, dict), f'unexpected json layout: {type(queries)}, {type(cands)}'
    if split == 'train' and MAX_TRAIN_QUERIES:
        queries = dict(list(queries.items())[:MAX_TRAIN_QUERIES])
    t0 = time.time()
    SPLITS[split] = {
        'q_ids':  {k: tokenize_case(v) for k, v in tqdm(queries.items(), desc=f'tokenize {split} queries')},          # query id  -> token ids
        'c_ids':  {k: tokenize_case(v) for k, v in tqdm(cands.items(), desc=f'tokenize {split} candidates')},            # candidate id -> token ids
        # real length of every query in words (before truncation), used in the analysis
        'q_words': {k: len((' '.join(v['text']) if isinstance(v['text'], list) else v['text']).split()) for k, v in queries.items()},
        # gold = relevant candidates that really are in this split's pool (empty for test: labels are hidden)
        'gold':   {k: [c for c in v.get('relevant_candidates', []) if c in cands] for k, v in queries.items()},
    }
    s = SPLITS[split]
    print(f'{split:5s}: {len(s["q_ids"])} queries | {len(s["c_ids"])} candidates | tokenized in {time.time() - t0:.0f}s | '
          f'mean query length {np.mean([len(v) for v in s["q_ids"].values()]):.0f} tokens | '
          f'mean #gold {np.mean([len(v) for v in s["gold"].values()]):.1f}')

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

tokenize train queries:   0%|          | 0/827 [00:00<?, ?it/s]

tokenize train candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

... (9 outputs left out)

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

tokenize test queries:   0%|          | 0/237 [00:00<?, ?it/s]

tokenize test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

test : 237 queries | 1727 candidates | tokenized in 3s | mean query length 1910 tokens | mean #gold 0.0


## 6. Retrieval model: chunking, pooling, similarity, evaluation

### Imports

In [22]:
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

### Retriever: token ids of ONE chunk -> ONE vector

In [23]:
class Retriever(nn.Module):
    """Wraps the Phase-1 model. forward(ids, lens) turns a batch of chunks into a batch of vectors.
       ids  : (B, L) token ids, right-padded with PAD_ID
       lens : (B,)   number of REAL tokens of every chunk"""

    def __init__(self, model, pooling='mean'):
        super().__init__()
        self.model = model
        self.pooling = pooling
        self.n_prompt = 0          # >0 when a soft prompt is attached: that many placeholder tokens are put in front

    def forward(self, ids, lens):
        B = ids.size(0)
        if self.n_prompt:          # placeholders: the soft-prompt embedding layer replaces them by the learned vectors
            ids = torch.cat([torch.full((B, self.n_prompt), PAD_ID, device=ids.device, dtype=ids.dtype), ids], dim=1)
        h, stride = encode_tokens(self.model, ids)                    # (B, L', d)
        h = h.float()
        # valid[b, t] = True for REAL text tokens (not padding, not the soft-prompt placeholders)
        pos = torch.arange(ids.size(1), device=ids.device)[None, :]
        valid = (pos >= self.n_prompt) & (pos < (self.n_prompt + lens)[:, None])
        valid = valid[:, ::stride][:, :h.size(1)]                     # the encoder of Arch 4 may shorten the sequence
        if self.pooling == 'mean':                                    # average of all real token vectors
            m = valid.unsqueeze(-1).float()
            return (h * m).sum(1) / m.sum(1).clamp(min=1)
        if self.pooling == 'last':                                    # vector of the last real token
            last = (valid.float() * torch.arange(1, valid.size(1) + 1, device=ids.device)).argmax(1)
            return h[torch.arange(B, device=ids.device), last]
        raise ValueError(self.pooling)

### Chunking: case -> chunks -> one case vector
A case with at most `CHUNK_LEN` tokens is one chunk. Longer cases are cut into consecutive chunks of `CHUNK_LEN` tokens;
the last chunk is moved back so that it also has `CHUNK_LEN` tokens (it overlaps a bit with the chunk before it).
The case vector is the **mean of its chunk vectors**.

In [24]:
def make_chunks(ids, n_chunks):
    ids = ids[:CHUNK_LEN * n_chunks]
    if len(ids) <= CHUNK_LEN:
        return [ids]
    starts = list(range(0, len(ids) - CHUNK_LEN + 1, CHUNK_LEN))
    if starts[-1] + CHUNK_LEN < len(ids):
        starts.append(len(ids) - CHUNK_LEN)
    return [ids[s:s + CHUNK_LEN] for s in starts]


def embed_cases(ret, cases, n_chunks, batch=CHUNK_BATCH_EVAL, desc=None):
    """cases: list of token-id lists -> tensor (N, d): one vector per case (differentiable if grads are enabled)."""
    chunks, owner = [], []
    for i, ids in enumerate(cases):
        for c in make_chunks(ids, n_chunks):
            chunks.append(c); owner.append(i)
    out = []
    for s in tqdm(range(0, len(chunks), batch), desc=desc, disable=desc is None, leave=False):   # progress bar only if desc is given
        part = chunks[s:s + batch]
        lens = torch.tensor([len(c) for c in part], device=device)
        x = torch.full((len(part), int(lens.max())), PAD_ID, dtype=torch.long)
        for j, c in enumerate(part):
            x[j, :len(c)] = torch.tensor(c)
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
            out.append(ret(x.to(device), lens))
    chunk_vec = torch.cat(out)                                       # (n_chunks_total, d)
    owner = torch.tensor(owner, device=device)
    case_vec = torch.zeros(len(cases), chunk_vec.size(1), device=device).index_add(0, owner, chunk_vec)
    return case_vec / torch.bincount(owner, minlength=len(cases)).unsqueeze(1).float()   # mean over the chunks

### Similarity, ranking and the Micro-F1@7 metric

In [25]:
def similarity(q, d, sim):
    """(Nq, dim) x (Nd, dim) -> (Nq, Nd) score matrix."""
    if sim == 'cosine':
        return F.normalize(q, dim=-1) @ F.normalize(d, dim=-1).T
    return (q @ d.T) / math.sqrt(q.size(-1))                         # plain dot product (scaled to keep numbers small)


def micro_prf(pred, gold, k=TOP_K):
    """Micro-averaged precision / recall / F1 @k.  pred: {qid: ranked list}, gold: {qid: list of relevant ids}.
    Only the first k predictions count; the gold set is NOT truncated or padded."""
    tp = sum(len(set(pred[q][:k]) & set(gold[q])) for q in gold)
    n_pred = sum(len(pred[q][:k]) for q in gold)
    n_gold = sum(len(gold[q]) for q in gold)
    p, r = tp / max(n_pred, 1), tp / max(n_gold, 1)
    return {'precision': p, 'recall': r, 'f1': 2 * p * r / max(p + r, 1e-12)}


@torch.no_grad()
def rank_split(ret, split, opt, keep_scores=False):
    """Embed all queries and the candidate pool of `split`, return for every query the candidates sorted by score."""
    ret.eval()
    S = SPLITS[split]
    q_keys, c_keys = list(S['q_ids']), list(S['c_ids'])
    q_vec = embed_cases(ret, [S['q_ids'][k] for k in q_keys], opt['n_chunks'], desc=f'encode {split} queries')
    c_vec = embed_cases(ret, [S['c_ids'][k] for k in c_keys], opt['n_chunks'], desc=f'encode {split} candidates')
    scores = similarity(q_vec, c_vec, opt['sim'])                    # (Nq, Nc)
    order = scores.argsort(dim=1, descending=True).cpu().numpy()
    ranked = {q: [c_keys[j] for j in order[i]] for i, q in enumerate(q_keys)}
    return (ranked, scores.cpu()) if keep_scores else ranked


def evaluate_dev(ret, opt):
    """returns (metrics, details). details = per query top-7, gold ids and the rank of every gold case (for error analysis)."""
    ranked = rank_split(ret, 'dev', opt)
    gold = SPLITS['dev']['gold']
    metrics = micro_prf(ranked, gold)
    details = {q: {'top7': ranked[q][:TOP_K], 'gold': gold[q],
                   'gold_ranks': [ranked[q].index(g) + 1 for g in gold[q]]} for q in ranked}
    return metrics, details


def predict_test(ret, opt):
    """top-7 candidate ids for every test query (checked against the rules of the task)."""
    ranked = rank_split(ret, 'test', opt)
    pool = set(SPLITS['test']['c_ids'])
    pred = {q: r[:TOP_K] for q, r in ranked.items()}
    for q, r in pred.items():
        assert len(r) == TOP_K and len(set(r)) == TOP_K and set(r) <= pool, f'bad prediction for {q}'
    return pred

## 7. PEFT techniques (hand-written, from `q4_peft.py`)

The four techniques of `q4_peft.py` were written for GPT-2 (`Conv1D` layers). The Phase-1 models use plain `nn.Linear`
layers with **separate** `q`, `k`, `v`, `o` matrices, so the same ideas are applied to those layers:

* **Soft prompt**: `m` trainable vectors that replace the embeddings of `m` placeholder tokens put in front of every chunk.
* **Adapter**: `h' = h + W_up GELU(W_down h + b_down) + b_up` after every block, `W_up, b_up` start at **zero** (identity at step 0).
* **LoRA**: `y = W0 x + (alpha/r) B A x`, `A` random, `B = 0` (so the update is zero at step 0).
* **LoRMA (LoRMA+)**: *multiplicative* update `W' = ((alpha/r) B A + I) W0`, which means `y = h + (alpha/r) B A h` with `h = W0 x`.
  `A, B` live in the **output** space of the matrix. `B = 0` at start, so again nothing changes at step 0.

In [26]:
# ---------------- soft prompt ----------------
class SoftPromptEmbedding(nn.Module):
    """Replaces model.tok. The first m positions of every input are placeholders; here they get the learned vectors."""

    def __init__(self, base_emb, init):                  # init: (m, d)
        super().__init__()
        self.base = base_emb                              # the frozen token embedding
        self.prompt = nn.Parameter(init.detach().clone().float())

    @property
    def weight(self):                                     # the (tied) LM head still looks for tok.weight
        return self.base.weight

    def forward(self, ids):
        e = self.base(ids)                                                    # (B, T, d)
        m = self.prompt.size(0)
        p = self.prompt.to(e.dtype).unsqueeze(0).expand(e.size(0), -1, -1)    # same prompt for every sequence
        return torch.cat([p, e[:, m:]], dim=1)                                # prompt FIRST, then the real tokens


# ---------------- adapter ----------------
class BottleneckAdapter(nn.Module):
    """h' = h + W_up GELU(W_down h + b_down) + b_up   (W_up and b_up are zero at the start -> identity map)."""

    def __init__(self, hidden, bottleneck=16):
        super().__init__()
        self.down = nn.Linear(hidden, bottleneck)
        self.up = nn.Linear(bottleneck, hidden)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


# ---------------- LoRA ----------------
class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B (A x).  W0 is frozen, A [r, in] is random, B [out, r] is zero."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        assert base.bias is None
        self.base, self.scaling = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))

    def forward(self, x):
        return self.base(x) + self.scaling * F.linear(F.linear(x, self.A), self.B)

    def merged(self):
        """One plain Linear with W = W0 + (alpha/r) B A  -> no extra cost at inference time."""
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=False).to(self.base.weight.device)
        with torch.no_grad():
            lin.weight.copy_(self.base.weight + self.scaling * (self.B @ self.A))
        return lin.requires_grad_(False)


# ---------------- LoRMA ----------------
def lorma_dense(x, W0, A, B, s):
    """Literal paper equation: build W' = (s B A + I) W0, then y = x W'^T."""
    W = (s * (B @ A) + torch.eye(W0.size(0), device=W0.device, dtype=W0.dtype)) @ W0
    return x @ W.T


class LoRMALinear(nn.Module):
    """LoRMA+ : W' = (s B A + I) W0.  Computed without ever building the big d x d matrix:
       h = x W0^T ;  y = h + s (h A^T) B^T.   A [r, out] is random, B [out, r] is zero."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        assert base.bias is None
        self.base, self.scaling = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))

    def forward(self, x):
        h = self.base(x)
        return h + self.scaling * F.linear(F.linear(h, self.A), self.B)

    def merged(self):
        """W' = W0 + s (B A) W0 : again a single plain Linear."""
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=False).to(self.base.weight.device)
        with torch.no_grad():
            lin.weight.copy_(self.base.weight + self.scaling * (self.B @ self.A) @ self.base.weight)
        return lin.requires_grad_(False)

### Attach a technique to the Phase-1 model, and merge LoRA / LoRMA

In [27]:
def wrap_linears(blocks, make):
    """Replace every nn.Linear called q / k / v / o inside `blocks` by make(linear)."""
    for blk in blocks:
        for mod in list(blk.modules()):                     # list(): we change the modules while looping
            for name in TARGETS:
                child = getattr(mod, name, None)
                if isinstance(child, nn.Linear):
                    setattr(mod, name, make(child))


def attach_peft(ret, method):
    """Freeze the whole Phase-1 model, then add the small trainable PEFT modules. Returns the list of trainable params."""
    for p in ret.parameters():                              # 1) freeze everything
        p.requires_grad_(False)
    model = ret.model                                       # 2) new modules are created AFTER freezing -> they are trainable
    blocks = get_blocks(model)
    if method == 'soft_prompt':
        ids = tokenizer.encode(SOFT_INIT_TEXT)
        ids = torch.tensor([ids[i % len(ids)] for i in range(SOFT_TOKENS)])   # cycle the words to fill m positions
        init = model.tok.weight.detach().float().cpu()[ids]                    # start from real word embeddings
        model.tok = SoftPromptEmbedding(model.tok, init.to(device))
        ret.n_prompt = SOFT_TOKENS
    elif method == 'adapter':
        ret.adapters = nn.ModuleList()                      # registered on `ret` so the optimizer finds the parameters
        for blk in blocks:
            ad = BottleneckAdapter(cfg.d_model, ADAPTER_BOTTLENECK).to(device)
            ret.adapters.append(ad)
            def hook(module, inputs, output, ad=ad):        # runs after the block; Arch 1 blocks return (x, v)
                return (ad(output[0]),) + tuple(output[1:]) if isinstance(output, tuple) else ad(output)
            blk.register_forward_hook(hook)
    elif method == 'lora':
        wrap_linears(blocks, lambda lin: LoRALinear(lin, RANK, ALPHA).to(device))
    elif method == 'lorma':
        wrap_linears(blocks, lambda lin: LoRMALinear(lin, RANK, ALPHA).to(device))
    else:
        raise ValueError(method)
    return [p for p in ret.parameters() if p.requires_grad]


def merge_peft(ret):
    """LoRA / LoRMA only: fold the update into the weights (used to measure inference speed without overhead)."""
    for mod in list(ret.modules()):
        for name, child in list(mod.named_children()):
            if isinstance(child, (LoRALinear, LoRMALinear)):
                setattr(mod, name, child.merged())

### Sanity checks (each technique starts from the Phase-1 function, parameter counts)

In [28]:
# 1) LoRMA: the cheap form must give the same output as the literal equation W' = (s B A + I) W0
_W0 = torch.randn(32, 20, dtype=torch.float64); _x = torch.randn(5, 20, dtype=torch.float64)
_A, _B = torch.randn(4, 32, dtype=torch.float64), torch.randn(32, 4, dtype=torch.float64)
_h = _x @ _W0.T
assert torch.allclose(lorma_dense(_x, _W0, _A, _B, 2.0), _h + 2.0 * (_h @ _A.T) @ _B.T)
print('LoRMA cheap form == literal equation: OK')

# 2) every technique must leave the model output unchanged at step 0 (except the soft prompt, which adds input tokens)
_ids = [SPLITS['dev']['q_ids'][k] for k in list(SPLITS['dev']['q_ids'])[:2]]
_ref = Retriever(build_phase1_model()).eval()
with torch.no_grad():
    _v0 = embed_cases(_ref, _ids, 2)
rows = []
for method in METHODS:
    r = Retriever(build_phase1_model())
    n_train = sum(p.numel() for p in attach_peft(r, method))
    r.eval()
    with torch.no_grad():
        diff = (embed_cases(r, _ids, 2) - _v0).abs().max().item()
    if method != 'soft_prompt':
        assert diff < 1e-2, f'{method} changes the model at step 0 (max diff {diff})'
    rows.append({'method': method, 'trainable params': n_train, '% of model': 100 * n_train / N_TOTAL_PARAMS,
                 'max |change| at step 0': diff})
    del r; gc.collect()
display(pd.DataFrame(rows))
del _ref

LoRMA cheap form == literal equation: OK


,method,trainable params,% of model,max |change| at step 0
0,soft_prompt,12288,0.006489,0.028383
1,adapter,202880,0.107135,0.000000
2,lora,327680,0.173039,0.000000
3,lorma,262144,0.138431,0.000000


## 8. Training

### Loss functions
For a batch of `B` queries we embed the queries `q_1..q_B` and a set of cases `d`. The first `B` cases `d_1..d_B` are the
**positives** (`d_i` is relevant to `q_i`). With `NEGATIVES='hard'` there are `B` more cases: one hard negative per query.
Every case that is not the positive of `q_i` is a negative of `q_i` (**in-batch negatives** + hard negatives).

* **InfoNCE** (contrastive / listwise): softmax over all cases, the positive must get the highest probability.
* **Margin** (pairwise): the positive must beat every negative by at least `MARGIN`.

In [29]:
def retrieval_loss(q, d, opt):
    """q: (B, dim) queries, d: (B or 2B, dim) cases. The positive of query i is d[i]."""
    B = q.size(0)
    if opt['sim'] == 'cosine':
        q, d = F.normalize(q, dim=-1), F.normalize(d, dim=-1)
    s = q @ d.T                                                    # (B, number of cases)
    if opt['sim'] == 'dot':
        s = s / math.sqrt(q.size(-1))
    target = torch.arange(B, device=q.device)
    if opt['loss'] == 'infonce':
        return F.cross_entropy(s / TAU if opt['sim'] == 'cosine' else s, target)
    pos = s[target, target].unsqueeze(1)                           # score of the positive of every query
    not_pos = torch.ones_like(s, dtype=torch.bool)
    not_pos[target, target] = False                                # the positive itself is not a negative
    viol = F.relu(MARGIN - pos + s) * not_pos                      # how much each negative violates the margin
    return viol.sum() / (B * (s.size(1) - 1))

### Zero-shot baseline and hard-negative mining (frozen Phase-1 model, no training)
The frozen model is used (1) as the **reference** to measure what training adds, and (2) to find **hard negatives**:
for every train query, the non-relevant train candidates that the frozen model ranks highest.
Hard negatives are drawn from the top `HARD_NEG_POOL` of that list (not just the very top one, because the very top
non-relevant case is sometimes actually relevant but unlabelled).

In [30]:
DEFAULT_OPT = {'pooling': POOLING, 'n_chunks': MAX_CHUNKS, 'sim': SIMILARITY, 'loss': LOSS, 'neg': NEGATIVES}

zs_model = Retriever(build_phase1_model(), POOLING)
ranked_train = rank_split(zs_model, 'train', DEFAULT_OPT)
HARD_NEG = {}
for q, ranking in ranked_train.items():
    gold = set(SPLITS['train']['gold'][q])
    HARD_NEG[q] = [c for c in ranking if c not in gold][:HARD_NEG_POOL]
zs_metrics, zs_details = evaluate_dev(zs_model, DEFAULT_OPT)
print('zero-shot (frozen Phase-1, mean pooling) dev:', {k: round(v, 4) for k, v in zs_metrics.items()})
del zs_model, ranked_train; gc.collect(); torch.cuda.empty_cache()

encode train queries:   0%|          | 0/101 [00:00<?, ?it/s]

encode train candidates:   0%|          | 0/503 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/15 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/120 [00:00<?, ?it/s]

zero-shot (frozen Phase-1, mean pooling) dev: {'precision': 0.0254, 'recall': 0.0216, 'f1': 0.0233}


### One experiment = fresh Phase-1 model + one PEFT technique + training + evaluation
After every epoch the dev set is evaluated; the epoch with the best dev F1 is kept. Then the test predictions are written
and the efficiency numbers (trainable parameters, peak GPU memory, training time, encoding latency) are measured.
Everything is saved in `OUTPUT_DIR/<run name>/result.json`; a run that is already saved is skipped.

In [31]:
def encoding_latency_ms(ret, n_chunks=16, repeats=3):
    """Time (ms) to encode 16 chunks of CHUNK_LEN tokens, i.e. the inference cost of the model."""
    ids = [SPLITS['dev']['q_ids'][k] for k in SPLITS['dev']['q_ids']]
    chunks = [c for x in ids for c in make_chunks(x, MAX_CHUNKS) if len(c) == CHUNK_LEN][:n_chunks]
    x = torch.tensor(chunks, device=device)
    lens = torch.full((len(chunks),), CHUNK_LEN, device=device)
    ret.eval()
    times = []
    with torch.no_grad():
        for i in range(repeats + 1):                      # first call is a warm-up
            if device == 'cuda': torch.cuda.synchronize()
            t = time.time()
            with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
                ret(x, lens)
            if device == 'cuda': torch.cuda.synchronize()
            if i: times.append((time.time() - t) * 1000)
    return float(np.mean(times))


def run_experiment(name, method, opt=None):
    run_dir = os.path.join(OUTPUT_DIR, name)
    result_path = os.path.join(run_dir, 'result.json')
    if os.path.exists(result_path):                       # finished earlier -> reuse
        print(f'[{name}] already done, loading saved result')
        return json.load(open(result_path))
    os.makedirs(run_dir, exist_ok=True)
    opt = {**DEFAULT_OPT, **(opt or {})}
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    ret = Retriever(build_phase1_model(), opt['pooling'])
    params = attach_peft(ret, method)
    n_train = sum(p.numel() for p in params)
    print(f'[{name}] {method}: {n_train:,} trainable parameters ({100 * n_train / N_TOTAL_PARAMS:.3f}% of the model) | {opt}')

    optimizer = torch.optim.AdamW(params, lr=PEFT_LR[method], weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and amp_dtype == torch.float16))
    S = SPLITS['train']
    train_q = [q for q in S['q_ids'] if S['gold'][q]]                   # queries that have at least one positive
    steps_per_epoch = len(train_q) // (BATCH_QUERIES * GRAD_ACCUM)
    total_steps = max(1, steps_per_epoch * EPOCHS)
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    t_start = time.time()

    best_f1, best_state, best_epoch, history, step = -1.0, None, 0, [], 0
    for epoch in range(EPOCHS):
        ret.train()
        random.shuffle(train_q)
        run_loss, n_loss = 0.0, 0
        bar = tqdm(range(steps_per_epoch), desc=f'[{name}] epoch {epoch + 1}/{EPOCHS}')   # one tick = one optimizer step
        for it in bar:
            # learning rate: short linear warm-up, then linear decay to 0
            warm = max(1, int(WARMUP_FRAC * total_steps))
            lr = PEFT_LR[method] * ((step + 1) / warm if step < warm else (total_steps - step) / (total_steps - warm))
            for g in optimizer.param_groups: g['lr'] = lr
            for m in range(GRAD_ACCUM):                                  # gradient accumulation = bigger effective batch
                b = (it * GRAD_ACCUM + m) * BATCH_QUERIES
                qs = train_q[b:b + BATCH_QUERIES]
                pos = [random.choice(S['gold'][q]) for q in qs]          # one random relevant case per query
                neg = [random.choice(HARD_NEG[q]) for q in qs] if opt['neg'] == 'hard' else []
                q_vec = embed_cases(ret, [S['q_ids'][q] for q in qs], opt['n_chunks'], batch=64)
                d_vec = embed_cases(ret, [S['c_ids'][c] for c in pos + neg], opt['n_chunks'], batch=64)
                loss = retrieval_loss(q_vec, d_vec, opt)
                scaler.scale(loss / GRAD_ACCUM).backward()
                run_loss += loss.item(); n_loss += 1
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            step += 1
            bar.set_postfix(loss=f'{run_loss / max(n_loss, 1):.4f}', lr=f'{lr:.1e}')
        metrics, _ = evaluate_dev(ret, opt)
        history.append({'epoch': epoch + 1, 'train_loss': run_loss / max(n_loss, 1), **metrics})
        print(f'   epoch {epoch + 1}/{EPOCHS} | loss {history[-1]["train_loss"]:.4f} | dev P {metrics["precision"]:.4f} '
              f'R {metrics["recall"]:.4f} F1 {metrics["f1"]:.4f} | {(time.time() - t_start) / 60:.1f} min')
        if metrics['f1'] > best_f1:                                       # keep the best epoch (only the tiny PEFT params)
            best_f1, best_epoch = metrics['f1'], epoch + 1
            best_state = {n: p.detach().cpu().clone() for n, p in ret.named_parameters() if p.requires_grad}
    train_minutes = (time.time() - t_start) / 60
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device == 'cuda' else 0.0

    # ---- final evaluation with the best epoch ----
    ret.load_state_dict(best_state, strict=False)
    torch.save(best_state, os.path.join(run_dir, 'peft_weights.pt'))     # the whole fine-tuned "delta" (tiny file)
    dev_metrics, dev_details = evaluate_dev(ret, opt)
    test_pred = predict_test(ret, opt)
    json.dump(test_pred, open(os.path.join(run_dir, 'prediction.json'), 'w'), indent=1)
    latency = encoding_latency_ms(ret)
    latency_merged = None
    if method in ('lora', 'lorma'):                                       # merged model has no extra operations
        merge_peft(ret)
        latency_merged = encoding_latency_ms(ret)

    result = {'name': name, 'method': method, 'opt': opt, 'trainable_params': n_train,
              'trainable_pct': 100 * n_train / N_TOTAL_PARAMS, 'best_epoch': best_epoch, 'dev': dev_metrics,
              'train_minutes': train_minutes, 'peak_train_mem_mb': peak_mb, 'latency_ms_16chunks': latency,
              'latency_ms_16chunks_merged': latency_merged, 'history': history, 'dev_details': dev_details}
    json.dump(result, open(result_path, 'w'))
    print(f'[{name}] DONE  dev F1 {dev_metrics["f1"]:.4f} (best epoch {best_epoch}) | {train_minutes:.1f} min | peak {peak_mb:.0f} MB')
    del ret, optimizer; gc.collect(); torch.cuda.empty_cache()
    return result

## 9. Experiments: the four PEFT techniques

In [32]:
RESULTS = {}
for method in METHODS:
    RESULTS[method] = run_experiment(method, method)

[soft_prompt] soft_prompt: 12,288 trainable parameters (0.006% of the model) | {'pooling': 'mean', 'n_chunks': 4, 'sim': 'cosine', 'loss': 'infonce', 'neg': 'hard'}


[soft_prompt] epoch 1/2:   0%|          | 0/51 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/15 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/120 [00:00<?, ?it/s]

   epoch 1/2 | loss 2.0880 | dev P 0.0303 R 0.0257 F1 0.0278 | 4.8 min


[soft_prompt] epoch 2/2:   0%|          | 0/51 [00:00<?, ?it/s]

... (43 outputs left out)

encode dev candidates:   0%|          | 0/120 [00:00<?, ?it/s]

encode test queries:   0%|          | 0/29 [00:00<?, ?it/s]

encode test candidates:   0%|          | 0/202 [00:00<?, ?it/s]

[lorma] DONE  dev F1 0.0245 (best epoch 2) | 9.7 min | peak 2078 MB


### Results table

In [33]:
zs_row = {'run': 'zero-shot (no training)', 'trainable params': 0, 'dev precision@7': zs_metrics['precision'],
          'dev recall@7': zs_metrics['recall'], 'dev F1@7': zs_metrics['f1']}
rows = [zs_row] + [{'run': r['name'], 'trainable params': r['trainable_params'], 'dev precision@7': r['dev']['precision'],
                    'dev recall@7': r['dev']['recall'], 'dev F1@7': r['dev']['f1']} for r in RESULTS.values()]
results_table = pd.DataFrame(rows)
display(results_table.round(4))
results_table.to_csv(os.path.join(OUTPUT_DIR, 'results_table.csv'), index=False)

,run,trainable params,dev precision@7,dev recall@7,dev F1@7
0,zero-shot (no training),0,0.0254,0.0216,0.0233
1,soft_prompt,12288,0.0303,0.0257,0.0278
2,adapter,202880,0.0278,0.0236,0.0256
3,lora,327680,0.0254,0.0216,0.0233
4,lorma,262144,0.0266,0.0226,0.0245


## 10. Analysis
The cells below produce the numbers and plots needed for the "Expected Analysis" of the task. The written discussion
is in the last markdown cell of the notebook.

### 10.1 Technique comparison and efficiency trade-off
(trainable parameters, training time, peak GPU memory during training, encoding latency; LoRA and LoRMA can be **merged** into
the weights after training, which removes their extra inference cost)

In [34]:
rows = [{'run': 'zero-shot', 'F1@7': zs_metrics['f1'], 'trainable params': 0, '% of model': 0.0, 'train min': 0.0,
         'train peak MB': 0.0, 'encode ms (16 chunks)': np.nan, 'encode ms merged': np.nan}]
for r in RESULTS.values():
    rows.append({'run': r['name'], 'F1@7': r['dev']['f1'], 'trainable params': r['trainable_params'],
                 '% of model': r['trainable_pct'], 'train min': r['train_minutes'], 'train peak MB': r['peak_train_mem_mb'],
                 'encode ms (16 chunks)': r['latency_ms_16chunks'],
                 'encode ms merged': r['latency_ms_16chunks_merged'] if r['latency_ms_16chunks_merged'] else np.nan})
eff = pd.DataFrame(rows)
display(eff.round(4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(eff['run'], eff['F1@7']); ax[0].set_ylabel('dev Micro-F1@7'); ax[0].set_title('Retrieval quality')
ax[0].tick_params(axis='x', rotation=30)
ax[1].scatter(eff['% of model'][1:], eff['F1@7'][1:])
for _, r in eff.iloc[1:].iterrows():
    ax[1].annotate(r['run'], (r['% of model'], r['F1@7']))
ax[1].axhline(zs_metrics['f1'], ls='--', c='gray', label='zero-shot'); ax[1].legend()
ax[1].set_xlabel('trainable parameters (% of the model)'); ax[1].set_ylabel('dev Micro-F1@7'); ax[1].set_title('Quality vs. size')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'technique_comparison.png'), dpi=150); plt.show()

,run,F1@7,trainable params,% of model,train min,train peak MB,encode ms (16 chunks),encode ms merged
0,zero-shot,0.0233,0,0.0000,0.0000,0.0000,NaN,NaN
1,soft_prompt,0.0278,12288,0.0065,9.5511,2046.9370,123.2688,NaN
2,adapter,0.0256,202880,0.1071,9.2655,2014.6504,119.5693,NaN
3,lora,0.0233,327680,0.1730,9.9824,2086.6191,126.4792,115.1188
4,lorma,0.0245,262144,0.1384,9.6965,2077.6035,122.4652,116.3987


### 10.2 Pooling and document representation (frozen Phase-1 model = how well do the Phase-1 representations transfer?)
No training here: we only change how the frozen model's vectors are pooled (**mean** vs **last token**) and how much of a long
case is read (**1 chunk** = the first `CHUNK_LEN` tokens vs **all chunks** = hierarchical mean of chunk vectors).

In [35]:
rows = []
for pooling in ('mean', 'last'):
    for n_chunks in (1, MAX_CHUNKS):
        o = {**DEFAULT_OPT, 'pooling': pooling, 'n_chunks': n_chunks}
        m, _ = evaluate_dev(Retriever(build_phase1_model(), pooling), o)
        rows.append({'pooling': pooling, 'chunks': n_chunks, 'precision@7': m['precision'], 'recall@7': m['recall'], 'F1@7': m['f1']})
        gc.collect(); torch.cuda.empty_cache()
zero_shot_grid = pd.DataFrame(rows)
display(zero_shot_grid.round(4))

encode dev queries:   0%|          | 0/4 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/32 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/15 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/120 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/4 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/32 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/15 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/120 [00:00<?, ?it/s]

,pooling,chunks,precision@7,recall@7,F1@7
0,mean,1,0.0254,0.0216,0.0233
1,mean,4,0.0254,0.0216,0.0233
2,last,1,0.0085,0.0072,0.0078
3,last,4,0.0085,0.0072,0.0078


### 10.3 Similarity function, ranking objective, negative sampling, pooling and chunking after training
Each row is one extra **LoRA** run (`ABLATION_METHOD`) in which exactly ONE thing differs from the default setting
(`mean` pooling, all chunks, cosine similarity, InfoNCE loss, in-batch + hard negatives). These are *ablations of the method*,
they are not counted as separate fine-tuning techniques.

In [36]:
ABLATIONS = [('default',          {}),
             ('pooling=last',     {'pooling': 'last'}),
             ('chunks=1',         {'n_chunks': 1}),
             ('sim=dot',          {'sim': 'dot'}),
             ('loss=margin',      {'loss': 'margin'}),
             ('negatives=inbatch', {'neg': 'inbatch'})]
ablation_rows = []
if RUN_ABLATIONS:
    for label, change in ABLATIONS:
        # the default ablation run IS the main LoRA run, so reuse it instead of training again
        r = RESULTS[ABLATION_METHOD] if label == 'default' else run_experiment(f'{ABLATION_METHOD}_abl_{label}', ABLATION_METHOD, change)
        ablation_rows.append({'setting': label, 'precision@7': r['dev']['precision'], 'recall@7': r['dev']['recall'],
                              'F1@7': r['dev']['f1'], 'train min': r['train_minutes'], 'peak MB': r['peak_train_mem_mb']})
    ablation_table = pd.DataFrame(ablation_rows)
    display(ablation_table.round(4))
    ablation_table.to_csv(os.path.join(OUTPUT_DIR, 'ablation_table.csv'), index=False)
    plt.figure(figsize=(7, 3.5)); plt.bar(ablation_table['setting'], ablation_table['F1@7'])
    plt.axhline(zs_metrics['f1'], ls='--', c='gray', label='zero-shot'); plt.legend()
    plt.xticks(rotation=30); plt.ylabel('dev Micro-F1@7'); plt.title(f'{ABLATION_METHOD}: one change at a time')
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'ablations.png'), dpi=150); plt.show()

[lora_abl_pooling=last] lora: 327,680 trainable parameters (0.173% of the model) | {'pooling': 'last', 'n_chunks': 4, 'sim': 'cosine', 'loss': 'infonce', 'neg': 'hard'}


[lora_abl_pooling=last] epoch 1/2:   0%|          | 0/51 [00:00<?, ?it/s]

encode dev queries:   0%|          | 0/15 [00:00<?, ?it/s]

encode dev candidates:   0%|          | 0/120 [00:00<?, ?it/s]

   epoch 1/2 | loss 2.9218 | dev P 0.0133 R 0.0113 F1 0.0122 | 4.9 min


[lora_abl_pooling=last] epoch 2/2:   0%|          | 0/51 [00:00<?, ?it/s]

... (57 outputs left out)

encode test queries:   0%|          | 0/29 [00:00<?, ?it/s]

encode test candidates:   0%|          | 0/202 [00:00<?, ?it/s]

[lora_abl_negatives=inbatch] DONE  dev F1 0.0245 (best epoch 2) | 7.1 min | peak 1538 MB


,setting,precision@7,recall@7,F1@7,train min,peak MB
0,default,0.0254,0.0216,0.0233,9.9824,2086.6191
1,pooling=last,0.0194,0.0164,0.0178,9.8852,2086.6040
2,chunks=1,0.0157,0.0134,0.0145,2.7518,1118.3418
3,sim=dot,0.0109,0.0092,0.0100,9.9647,2086.6191
4,loss=margin,0.0303,0.0257,0.0278,9.9422,2086.6191
5,negatives=inbatch,0.0266,0.0226,0.0245,7.0574,1538.1699


### 10.4 Retrieval errors, failure patterns and long documents
Uses the dev predictions of the best technique. We look at: how many queries get no hit at all, where the relevant cases land in the
ranking, whether short or long queries are harder, and whether the model prefers long candidates.

In [37]:
best_name = max(RESULTS, key=lambda k: RESULTS[k]['dev']['f1'])
details = RESULTS[best_name]['dev_details']
print('best technique on dev:', best_name)

q_len = {q: SPLITS['dev']['q_words'][q] for q in details}              # length in words, before truncation
hits = {q: len(set(d['top7']) & set(d['gold'])) for q, d in details.items()}
n_gold = {q: len(d['gold']) for q, d in details.items()}
all_ranks = [r for d in details.values() for r in d['gold_ranks']]

print(f'queries with 0 hits in the top 7 : {sum(h == 0 for h in hits.values())} / {len(hits)}')
print(f'queries with >= 1 hit            : {sum(h > 0 for h in hits.values())} / {len(hits)}')
print(f'gold cases ranked in top 7 / 20 / 50 : {np.mean(np.array(all_ranks) <= 7):.2%} / {np.mean(np.array(all_ranks) <= 20):.2%} / {np.mean(np.array(all_ranks) <= 50):.2%}')
print(f'median rank of a relevant case   : {np.median(all_ranks):.0f} (pool size {len(SPLITS["dev"]["c_ids"])})')
print(f'gold cases per query             : mean {np.mean(list(n_gold.values())):.1f}, max {max(n_gold.values())}  '
      f'(top 7 can never recall more than 7, so many gold cases per query lowers recall)')

# do long queries behave differently? (query length in words; the three groups have the same number of queries)
df = pd.DataFrame({'q_len': pd.Series(q_len), 'hits': pd.Series(hits), 'n_gold': pd.Series(n_gold)})
df['recall@7'] = df['hits'] / df['n_gold'].clip(lower=1)
df['length bin'] = pd.qcut(df['q_len'].rank(method='first'), 3, labels=['short', 'medium', 'long'])   # 3 equal-sized groups
display(df.groupby('length bin', observed=True)[['recall@7', 'hits']].mean().round(3))
print('fraction of queries that are cut by the window :', np.mean([len(v) >= MAX_TOKENS for v in SPLITS['dev']['q_ids'].values()]).round(3))

# length bias: are the wrongly retrieved candidates longer / shorter than the correct ones?
c_len = {c: len(v) for c, v in SPLITS['dev']['c_ids'].items()}
wrong = [c_len[c] for d in details.values() for c in d['top7'] if c not in d['gold']]
right = [c_len[c] for d in details.values() for c in d['top7'] if c in d['gold']]
print(f'mean length (tokens) of correct top-7 hits {np.mean(right):.0f} | of wrong top-7 results {np.mean(wrong):.0f} | all candidates {np.mean(list(c_len.values())):.0f}')

plt.figure(figsize=(6, 3.5)); plt.hist(np.clip(all_ranks, 1, 200), bins=40)
plt.axvline(TOP_K, c='r', ls='--', label='top-7 cut-off'); plt.legend()
plt.xlabel('rank of a relevant case (clipped at 200)'); plt.ylabel('count'); plt.title(f'Where do relevant cases land? ({best_name})')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'gold_rank_hist.png'), dpi=150); plt.show()

best technique on dev: soft_prompt
queries with 0 hits in the top 7 : 100 / 118
queries with >= 1 hit            : 18 / 118
gold cases ranked in top 7 / 20 / 50 : 2.57% / 6.47% / 13.98%
median rank of a relevant case   : 240 (pool size 1023)
gold cases per query             : mean 8.2, max 92  (top 7 can never recall more than 7, so many gold cases per query lowers recall)


,recall@7,hits
length bin,,
short,0.063,0.154
medium,0.037,0.179
long,0.019,0.300


fraction of queries that are cut by the window : 0.89
mean length (tokens) of correct top-7 hits 1928 | of wrong top-7 results 1910 | all candidates 1858


## 11. Final prediction file (best technique on dev)

In [38]:
# The test predictions of every run were already written in its folder. We submit the best technique on dev.
final = json.load(open(os.path.join(OUTPUT_DIR, best_name, 'prediction.json')))
pool = set(SPLITS['test']['c_ids'])
assert set(final) == set(SPLITS['test']['q_ids']), 'every test query needs a prediction'
assert all(len(v) == TOP_K and len(set(v)) == TOP_K and set(v) <= pool for v in final.values())
pred_path = os.path.join(OUTPUT_DIR, 'prediction.json')
json.dump(final, open(pred_path, 'w'), indent=1)
with zipfile.ZipFile(os.path.join(OUTPUT_DIR, 'submission.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname='prediction.json')
print(f'Saved prediction.json ({len(final)} test queries, technique: {best_name}) and submission.zip')

Saved prediction.json (237 test queries, technique: soft_prompt) and submission.zip


In [39]:
import shutil
import os

folder = "/kaggle/working/phase7_arch4_encdec"
output = "/kaggle/working/phase7_arch4_encdec"

shutil.make_archive(output, 'zip', folder)

print(f"ZIP created: {output}.zip")
print(f"Size: {os.path.getsize(output + '.zip') / (1024**3):.2f} GB")

ZIP created: /kaggle/working/phase7_arch4_encdec.zip
Size: 0.01 GB


## 12. Written analysis
*Read the tables and plots above, then complete / edit the points below with your own numbers. Each point says where the evidence is.*

**1. How do the adaptation techniques affect retrieval quality?**  -> table in 9 and 10.1.
Compare the four techniques with each other and with the zero-shot row. Things to note: soft prompt trains only 16 x 768 = 12,288 numbers and
can only change the *input*; the adapter adds a small non-linear block after every layer; LoRA and LoRMA change the *attention matrices*
themselves (additively vs. multiplicatively), so they can reshape what the model attends to when it builds the case vector.

**2. How well do the Phase-1 legal representations transfer to PCR?**  -> zero-shot row, 10.2.
The Phase-1 model was trained only for **next-token prediction**. Its hidden states are good at predicting the next word, not
necessarily at placing similar cases close together, so the zero-shot score shows how much case-similarity information is already
there, and the gap to the trained runs shows what the contrastive PEFT training adds.
For this architecture the encoder (bidirectional) is used, which sees the whole chunk at once; the decoder-only models only see left context.

**3. Effect of pooling and document representation?**  -> 10.2 (frozen) and 10.3 (`pooling=last`, `chunks=1`).

**4. Effect of the similarity function and ranking objective?**  -> 10.3 (`sim=dot`, `loss=margin`).
Cosine similarity ignores vector length; the raw dot product is also influenced by the vector norm (which can depend on the case length).
InfoNCE looks at all candidates at once; the pairwise margin loss only asks "positive better than negative by a margin".

**5. Impact of hard negatives / negative sampling?**  -> 10.3 (`negatives=inbatch`).
In-batch negatives are mostly *easy* (random cases). Hard negatives are cases that look similar but are not relevant,
which is exactly what the test has to separate. Risk: a hard negative may be an unlabelled relevant case (false negative).

**6. Challenges of long legal documents?**  -> 10.2 / 10.3 (`chunks=1` vs all chunks), 10.4.
Cases are far longer than one chunk. We only read the first `MAX_TOKENS` tokens, so anything later is invisible; averaging chunk vectors
blurs different parts of a case into one vector.

**7. Retrieval errors and common failure patterns?**  -> 10.4 (zero-hit queries, rank histogram, length bins, length bias).

**8. Efficiency trade-off (quality, size, memory, inference)?**  -> 10.1.
All techniques store only a tiny file (`peft_weights.pt`) per task. LoRA and LoRMA can be merged into the weights (no extra
inference cost, see `encode ms merged`), the adapter and the soft prompt always add computation (extra layers / extra tokens).
Training memory is dominated by the frozen backbone's activations, not by the optimizer state, so PEFT saves less memory than parameters.

**9. Why might a technique that works on another legal NLP task not transfer to PCR?**
PCR is not classification: the model must produce a *vector space* in which a whole long case lands near its relevant cases, and the
score is decided by ranking against a large pool. A technique that is enough to move a single class-decision boundary (soft prompt, a
small adapter) may be too weak to reshape the whole embedding geometry, and anything that depends on the length of the input
(prompt position, truncation) interacts with the very long documents of PCR.

*Knowledge distillation was intentionally not used in this notebook.*